## Token Intervention with Fixed Reasoning

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Patches a counterfactual number into a truncated reasoning chain while keeping the rest of the chain fixed to the model's own original continuation. Tests whether a short continuation right after the swap still shows signs the model reacted to the counterfactual number.

### Set-up

In [2]:
import sys
sys.path.append("src")

import torch
import gc
from tqdm import tqdm

import _config

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS", # GPT-OSS or R1
    prompt_type="h_pre_result_2", # empty or pre_result or pre_sum
)
run_config = _config.RunConfig(
    experiment_root="experiments/token_intervention",
    output_filename=f"fixed_reasoning{prompt_config.suffix}.csv",
    overwrite=True,
)
batch_size = 22

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
divided_prompts = _config.load_divided_prompts(prompt_config)
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 2816 divided prompts


In [8]:
def get_fixed_reasoning_intervention_prompt(row):
    return row['base_before'] + str(row['source_number']) + row['base_after']

# Batch code below uses _config.build_number_prompts for the same construction.

### Run intervention + short continuation

For each divided prompt, swap in the counterfactual number but keep the rest of the chain factual, then generate a short continuation and check whether it changes.

In [11]:
# Get header of divided prompts dataset
header = list(divided_prompts.columns) + ['intervention_prompt', 'generated_text']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(divided_prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = divided_prompts.iloc[i:i+batch_size]
    
    # Prepare batch of intervention prompts
    intervention_prompts = _config.build_number_prompts(batch_rows, "source_number", after_column="base_after")
    
    # Tokenize all prompts in the batch
    tokenized_inputs = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    
    # Generate for the entire batch
    generations = model.generate(
        tokenized_inputs.input_ids,
        attention_mask=tokenized_inputs.attention_mask,
        max_new_tokens=3,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(generations[j]).replace(intervention_prompt, "").replace(tokenizer.pad_token[-1], "")
        _config.write_to_csv(filepath, row.to_list() + [intervention_prompt, generated_text])


  0%|                                                                                         | 0/128 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████| 128/128 [08:43<00:00,  4.09s/it]
